# Text Chunker
Splits annual report text into ESG-relevant chunks ready for Gemini scoring.
Also filters out Inline XBRL metadata noise.

In [13]:
ESG_KEYWORDS = [
    # Environmental
    "carbon", "emission", "climate", "greenhouse", "renewable", "energy",
    "waste", "water", "pollution", "biodiversity", "sustainability", "net zero",
    "fossil", "solar", "wind", "environmental",
    # Social
    "employee", "diversity", "inclusion", "gender", "health", "safety",
    "community", "human rights", "labour", "labor", "training", "wellbeing",
    "workforce", "social",
    # Governance
    "board", "governance", "executive", "compensation", "audit", "compliance",
    "ethics", "transparency", "shareholder", "director", "risk management",
    "anti-corruption", "whistleblower",
]
print(f"Loaded {len(ESG_KEYWORDS)} ESG keywords.")


Loaded 43 ESG keywords.


In [14]:
def is_xbrl_noise(chunk: str) -> bool:
    """
    Returns True if the chunk looks like Inline XBRL metadata.
    Filters on both ratio (>15%) AND absolute count (>8 XBRL tokens).
    """
    words = chunk.split()
    if not words:
        return True
    xbrl_count = sum(
        1 for w in words
        if "us-gaap:" in w
        or "fasb.org" in w
        or "xbrli:" in w
        or "bac:" in w
        or "srt:" in w
        or "iso4217:" in w
        or "utr:" in w
        or "stpr:" in w
        or "country:" in w
        or (len(w) > 25 and "Member" in w)
        or (len(w) > 20 and w.count(":") >= 2)
        or (w.startswith("0000") and len(w) in (7, 10))  # CIK numbers
    )
    ratio = xbrl_count / len(words)
    # Noise if ratio high OR too many absolute XBRL tokens
    return ratio > 0.15 or xbrl_count > 8

def split_into_chunks(text: str, chunk_size: int = 500, overlap: int = 50) -> list:
    words  = text.split()
    chunks = []
    start  = 0
    while start < len(words):
        chunks.append(" ".join(words[start : start + chunk_size]))
        start += chunk_size - overlap
    return chunks

def is_esg_relevant(chunk: str) -> bool:
    lower = chunk.lower()
    return any(kw in lower for kw in ESG_KEYWORDS)

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list:
    all_chunks = split_into_chunks(text, chunk_size, overlap)
    clean      = [c for c in all_chunks if not is_xbrl_noise(c)]
    relevant   = [c for c in clean if is_esg_relevant(c)]
    print(f"  Total: {len(all_chunks)} → after XBRL filter: {len(clean)} → ESG relevant: {len(relevant)}")
    return relevant

print("Functions defined.")


Functions defined.


In [15]:
sample = """
The company committed to reducing carbon emissions by 50% by 2030.
Our board approved new governance policies this year.
Revenue increased 12% driven by product sales in Asia.
We invested in renewable energy across all facilities.
Employee diversity programs were expanded with inclusion initiatives.
""" * 20

chunks = chunk_text(sample, chunk_size=30, overlap=5)
print(f"\nFirst chunk:\n{chunks[0]}")


  Total: 36 → after XBRL filter: 36 → ESG relevant: 36

First chunk:
The company committed to reducing carbon emissions by 50% by 2030. Our board approved new governance policies this year. Revenue increased 12% driven by product sales in Asia. We invested
